# HW 4: Text Classification
### COSC 426: Fall 2025, Colgate University

Use this notebook to answer questions and load + display from your experiments. Feel free to add as many code and markdown chunks as you would like in each of the sub-sections. 

**If you use any external resources (e.g., code snippets, reference articles), please cite them in comments or text!**

## Part 1: Train and evaluate `distilgpt` based Bayesian Classifier

In this part, load in the probability estimates from your finetuned **language models**, use it to classify text, and display classification results. 

1. Data Preparation: Sample balanced subsets from IMDB to get the class-conditional datasets.

2. Training Two distilGPT Models: Model A trained on pos sentences and Model B trained on neg sentences

3. Classification Rule: The classifier will give the final conclusion based on 
$$P(Class∣text) \stackrel{\propto}{\sim} P(text∣Class)P(Class)$$
- For each review:
    - Compute average log-likelihood under the positive model
    - Compute average log-likelihood under the negative model
    - Predict the class corresponding to the higher log-likelihood (as the priors are same).

In [19]:
from datasets import load_dataset
import random, os

ds = load_dataset("imdb")

def save_subset(split, n_per_class, out_prefix):
    data = ds[split]
    pos = [ex["text"] for ex in data if ex["label"] == 1]
    neg = [ex["text"] for ex in data if ex["label"] == 0]

    random.seed(42)
    pos_s = random.sample(pos, n_per_class)
    neg_s = random.sample(neg, n_per_class)

    os.makedirs("data", exist_ok=True)

    with open(f"data/{out_prefix}_pos.tsv", "w") as f:
        f.write("text\n")
        for t in pos_s:
            clean_t = t.replace("\t", " ")
            f.write(f"{clean_t}\n")

    with open(f"data/{out_prefix}_neg.tsv", "w") as f:
        f.write("text\n")
        for t in neg_s:
            clean_t = t.replace("\t", " ")
            f.write(f"{clean_t}\n")

save_subset("train", 2500, "train")
save_subset("test", 1000, "test")

In [26]:
def save_test_classification(split, n_per_class, out_prefix):
    data = ds[split]
    pos = [ex['text'] for ex in data if ex['label'] == 1]
    neg = [ex['text'] for ex in data if ex['label'] == 0]
    
    random.seed(42) 
    pos_s = random.sample(pos, n_per_class)
    neg_s = random.sample(neg, n_per_class)

    os.makedirs("data", exist_ok=True)
    with open(f"data/{out_prefix}.tsv", "w") as f:
        f.write("sentid\tsentence\ttarget\n")
        i = 1
        for t in pos_s:
            clean_t = t.replace("\t", " ")
            f.write(f"{i}\t{clean_t}\t1\n")
            i += 1
        for t in neg_s:
            clean_t = t.replace("\t", " ")
            f.write(f"{i}\t{clean_t}\t0\n")
            i += 1
            
save_test_classification("test", 1000, "test_classification_1")

In [51]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

df_negmodel = pd.read_csv("predictions/test_1_neg.tsv", sep="\t")
df_posmodel = pd.read_csv("predictions/test_1_pos.tsv", sep="\t")

# Compute sentence log-likelihoods for each model
results_neg = df_negmodel.groupby("sentid").agg(
    log_likelihood_neg=("prob", lambda x: np.sum(np.log2(x)))
)
results_pos = df_posmodel.groupby("sentid").agg(
    log_likelihood_pos=("prob", lambda x: np.sum(np.log2(x)))
)

results = results_pos.join(results_neg)

# Skip priors for simplicity as they are equal

results["predicted"] = np.where(
    results["log_likelihood_pos"] > results["log_likelihood_neg"],
    1,   
    0    
)

df_gold = pd.read_csv(
    "/Users/huxiyan/Desktop/H/25Fall/COSC426/cosc426/hw/hw04/data/test_classification_1.tsv",
    sep="\t"
)

results = results.merge(df_gold[["sentid", "target"]], left_index=True, right_on="sentid")

y_true_1 = results["target"].tolist()
y_pred_1 = results["predicted"].tolist()

# Evaluate
accuracy = accuracy_score(y_true_1, y_pred_1)
precision, recall, f1, _ = precision_recall_fscore_support(y_true_1, y_pred_1)

print(f"Accuracy: {accuracy:.4f}\n")
print(f"Precision (positive class): {precision[1]:.4f}")
print(f"Recall (positive class): {recall[1]:.4f}")
print(f"F1-score (positive class): {f1[1]:.4f}\n")
print(f"Precision (negative class): {precision[0]:.4f}")
print(f"Recall (negative class): {recall[0]:.4f}")
print(f"F1-score (negative class): {f1[0]:.4f}")


Accuracy: 0.8710

Precision (positive class): 0.8865
Recall (positive class): 0.8510
F1-score (positive class): 0.8684

Precision (negative class): 0.8567
Recall (negative class): 0.8910
F1-score (negative class): 0.8735


In [53]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

df_negmodel = pd.read_csv("predictions/test_1_neg_full.tsv", sep="\t")
df_posmodel = pd.read_csv("predictions/test_1_pos_full.tsv", sep="\t")

# Compute sentence log-likelihoods for each model
results_neg = df_negmodel.groupby("sentid").agg(
    log_likelihood_neg=("prob", lambda x: np.sum(np.log2(x)))
)
results_pos = df_posmodel.groupby("sentid").agg(
    log_likelihood_pos=("prob", lambda x: np.sum(np.log2(x)))
)

results = results_pos.join(results_neg)

# Skip priors for simplicity as they are equal

results["predicted"] = np.where(
    results["log_likelihood_pos"] > results["log_likelihood_neg"],
    1,   
    0    
)

df_gold = pd.read_csv(
    "/Users/huxiyan/Desktop/H/25Fall/COSC426/cosc426/hw/hw04/data/test_classification_1_full.tsv",
    sep="\t"
)

results = results.merge(df_gold[["sentid", "target"]], left_index=True, right_on="sentid")

y_true_1 = results["target"].tolist()
y_pred_1 = results["predicted"].tolist()

# Evaluate
accuracy = accuracy_score(y_true_1, y_pred_1)
precision, recall, f1, _ = precision_recall_fscore_support(y_true_1, y_pred_1)

print(f"Accuracy: {accuracy:.4f}\n")
print(f"Precision (positive class): {precision[1]:.4f}")
print(f"Recall (positive class): {recall[1]:.4f}")
print(f"F1-score (positive class): {f1[1]:.4f}\n")
print(f"Precision (negative class): {precision[0]:.4f}")
print(f"Recall (negative class): {recall[0]:.4f}")
print(f"F1-score (negative class): {f1[0]:.4f}")

Accuracy: 0.8994

Precision (positive class): 0.9154
Recall (positive class): 0.8802
F1-score (positive class): 0.8975

Precision (negative class): 0.8847
Recall (negative class): 0.9186
F1-score (negative class): 0.9013


## Part 2: Train and evaluate a `distilgpt` based TextClassification model
In this part, load in and display the results from your finetuned **TextClassification models**. 

In [14]:
def save_classification(split, n_per_class, out_prefix):
    data = ds[split]
    pos = [ex['text'] for ex in data if ex['label'] == 1]
    neg = [ex['text'] for ex in data if ex['label'] == 0]
    
    # Same data for Part1 and Part2 models
    random.seed(42) 
    pos_s = random.sample(pos, n_per_class)
    neg_s = random.sample(neg, n_per_class)

    os.makedirs("data", exist_ok=True)
    with open(f"data/{out_prefix}.tsv", "w") as f:
        f.write("sentid\ttext\tlabel\n")
        i = 0
        for t in pos_s:
            clean_t = t.replace("\t", " ")
            f.write(f"{i}\t{clean_t}\t1\n")
            i += 1
        for t in neg_s:
            clean_t = t.replace("\t", " ")
            f.write(f"{i}\t{clean_t}\t0\n")
            i += 1

save_classification("train", 2500, "train_classification")
save_classification("test", 1000, "test_classification")

In [39]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

df_textclassification = pd.read_csv("predictions/test_2.tsv", sep="\t")

y_true_2 = df_textclassification["target"].tolist()
y_pred_2 = df_textclassification["predicted"].tolist()
y_pred_2 = [1 if p == "Positive" else 0 for p in y_pred_2]

accuracy = accuracy_score(y_true_2, y_pred_2)
precision, recall, f1, _ = precision_recall_fscore_support(y_true_2, y_pred_2)

print(f"Accuracy: {accuracy:.4f}\n")
print(f"Precision (positive class): {precision[1]:.4f}")
print(f"Recall (positive class): {recall[1]:.4f}")
print(f"F1-score (positive class): {f1[1]:.4f}\n")
print(f"Precision (negative class): {precision[0]:.4f}")
print(f"Recall (negative class): {recall[0]:.4f}")
print(f"F1-score (negative class): {f1[0]:.4f}")

Accuracy: 0.9040

Precision (positive class): 0.8892
Recall (positive class): 0.9230
F1-score (positive class): 0.9058

Precision (negative class): 0.9200
Recall (negative class): 0.8850
F1-score (negative class): 0.9021


In [38]:
# Train and test on the entire dataset
df_textclassification_full = pd.read_csv("predictions/test_2_full.tsv", sep="\t")

y_true = df_textclassification_full["target"].tolist()
y_pred = df_textclassification_full["predicted"].tolist()
y_pred = [1 if p == "Positive" else 0 for p in y_pred]

accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred)

print(f"Accuracy: {accuracy:.4f}\n")
print(f"Precision (positive class): {precision[1]:.4f}")
print(f"Recall (positive class): {recall[1]:.4f}")
print(f"F1-score (positive class): {f1[1]:.4f}\n")
print(f"Precision (negative class): {precision[0]:.4f}")
print(f"Recall (negative class): {recall[0]:.4f}")
print(f"F1-score (negative class): {f1[0]:.4f}")

Accuracy: 0.9329

Precision (positive class): 0.9208
Recall (positive class): 0.9472
F1-score (positive class): 0.9338

Precision (negative class): 0.9456
Recall (negative class): 0.9186
F1-score (negative class): 0.9319


## Part 3: Reflect on the two approaches to classification

In this part, answer the question in `HW4.md` in markdown chunks. If you used external sources to find and make sense of this, please cite them!

In **Part 1**, the classifier is built using **Bayes’ rule** and **word-level probabilities**. The model assumes that words are generated independently from class-specific distributions, and classification is done by computing:  

$$
P(\text{Class} \mid \text{text}) \stackrel{\propto}{\sim} P(\text{Class}) \prod_i P(w_i \mid \text{Class})
$$

Here, the language model is used to estimate these probabilities, but the classification itself is not trained end-to-end. We are computing likelihoods from a pretrained model and applying Bayes’ rule manually.

In **Part 2**, instead of using Bayes’ rule explicitly, the model learns directly to predict the class label (positive or negative) by adjusting its weights through gradient descent. The entire transformer learns representations that separate the classes based on training data.

## Part 4 (Optional): Error analysis comparing the two approaches to classification

In [48]:
# Bayesian Classifier
results["correct"] = results["predicted"] == results["target"]
misclassified_bayes = results[~results["correct"]]

# print("Misclassified sentences in Bayesian classifier:")
# print(misclassified_bayes[["sentid", "predicted", "target"]])

# TextClassification Model
df_textclassification["predicted_binary"] = [1 if p == "Positive" else 0 for p in df_textclassification["predicted"]]
df_textclassification["correct"] = df_textclassification["predicted_binary"] == df_textclassification["target"]
df_textclassification = df_textclassification.rename(columns={"textid": "sentid"})
misclassified_textclf = df_textclassification[~df_textclassification["correct"]]

# print("\Misclassified sentences in TextClassification model:")
# print(misclassified_textclf[["sentid", "predicted", "target"]])

# Sentences both models got wrong
shared_misclassified = pd.merge(
    misclassified_bayes[["sentid"]],
    misclassified_textclf[["sentid"]],
    on="sentid"
)
print(f"\n Sentences misclassified by BOTH models: {len(shared_misclassified)}")
# print(shared_misclassified)

# Sentences ONLY misclassified by Bayesian model
only_bayes = misclassified_bayes[~misclassified_bayes["sentid"].isin(shared_misclassified["sentid"])]
print(f"\nSentences ONLY misclassified by Bayesian model: {len(only_bayes)}")
print(only_bayes["sentid"].tolist())

# Sentences ONLY misclassified by TextClassification model
only_textclf = misclassified_textclf[~misclassified_textclf["sentid"].isin(shared_misclassified["sentid"])]
print(f"\nSentences ONLY misclassified by TextClassification model: {len(only_textclf)}")
print(only_textclf["sentid"].tolist())


 Sentences misclassified by BOTH models: 39

Sentences ONLY misclassified by Bayesian model: 332
[1, 3, 4, 12, 23, 41, 47, 51, 53, 58, 60, 64, 65, 76, 80, 90, 95, 102, 115, 121, 123, 126, 128, 131, 133, 135, 136, 148, 156, 162, 167, 170, 180, 181, 185, 187, 192, 200, 205, 207, 209, 219, 228, 229, 234, 237, 238, 241, 245, 246, 247, 250, 256, 257, 259, 269, 274, 277, 279, 283, 286, 303, 309, 316, 322, 325, 337, 347, 350, 361, 362, 364, 369, 375, 377, 382, 389, 390, 392, 395, 424, 439, 443, 451, 463, 466, 469, 476, 479, 483, 485, 491, 496, 497, 503, 509, 518, 520, 525, 527, 534, 537, 540, 543, 544, 546, 556, 565, 569, 584, 585, 587, 589, 599, 601, 608, 613, 614, 615, 617, 621, 622, 625, 626, 628, 631, 645, 647, 650, 653, 655, 660, 663, 669, 675, 678, 686, 688, 690, 692, 693, 696, 703, 709, 711, 712, 716, 724, 726, 734, 735, 738, 740, 742, 748, 750, 751, 754, 758, 759, 765, 767, 769, 770, 771, 776, 798, 805, 822, 824, 827, 838, 845, 855, 858, 859, 870, 873, 875, 877, 885, 889, 890, 896, 8

In [50]:
df_test = pd.read_csv(
    "/Users/huxiyan/Desktop/H/25Fall/COSC426/cosc426/hw/hw04/data/test_classification_1.tsv",
    sep="\t"
)

df_test["length"] = df_test["sentence"].apply(lambda x: len(str(x).split()))

avg_len_all = df_test["length"].mean()
print(f"Average sentence length (all): {avg_len_all:.2f} words")

# Average sentence length for misclassified by Bayesian model
mis_bayes_ids = only_bayes["sentid"]
avg_len_bayes = df_test[df_test["sentid"].isin(mis_bayes_ids)]["length"].mean()
print(f"Average sentence length (misclassified by Bayesian): {avg_len_bayes:.2f} words")

# Average sentence length for misclassified by TextClassification model
mis_textclf_ids = only_textclf["sentid"]
avg_len_textclf = df_test[df_test["sentid"].isin(mis_textclf_ids)]["length"].mean()
print(f"Average sentence length (misclassified by TextClassification): {avg_len_textclf:.2f} words")

Average sentence length (all): 224.44 words
Average sentence length (misclassified by Bayesian): 215.35 words
Average sentence length (misclassified by TextClassification): 236.18 words



The results from both approaches show a clear improvement in performance when moving from the Bayesian classifier to the `distilgpt` text classification model. 

The **Bayesian classifier** achieves accuracy about 81%, but its performance is notably lower than the GPT classifier. Since it assumes that words occur independently, it cannot capture complex contextual relationships between words. This leads to misclassifications especially when sentiment depends on multi-word expressions or subtle phrasing.

In contrast, the **GPT classifier** learns contextualized representations of text through its transformer layers. It captures long-range dependencies and nuanced sentiment cues, resulting in higher precision, recall, and F1-scores across both classes. 

Interestingly, the sentence-length analysis supports these observations. Sentences misclassified by the Bayesian model are slightly shorter on average, suggesting that its errors are not limited to overly complex or long inputs—it may fail due to missing contextual understanding even in moderately sized sentences. The TextClassification model’s misclassified sentences are longer on average , indicating that it generally handles shorter texts well but may struggle with very long reviews where sentiment shifts or mixed opinions appear.

In [24]:
# Entire dataset without subsampling
from datasets import load_dataset
import os

ds = load_dataset("imdb")

def save_subset(split, out_prefix):
    data = ds[split]
    pos = [ex["text"] for ex in data if ex["label"] == 1]
    neg = [ex["text"] for ex in data if ex["label"] == 0]

    os.makedirs("data", exist_ok=True)

    with open(f"data/{out_prefix}_pos.tsv", "w") as f:
        f.write("text\n")
        for t in pos:
            clean_t = t.replace("\t", " ")
            f.write(f"{clean_t}\n")

    with open(f"data/{out_prefix}_neg.tsv", "w") as f:
        f.write("text\n")
        for t in neg:
            clean_t = t.replace("\t", " ")
            f.write(f"{clean_t}\n")

save_subset("train", "train_full")
save_subset("test", "test_full")

def save_classification_full(split, out_prefix):
    data = ds[split]
    pos = [ex['text'] for ex in data if ex['label'] == 1]
    neg = [ex['text'] for ex in data if ex['label'] == 0]

    os.makedirs("data", exist_ok=True)
    with open(f"data/{out_prefix}.tsv", "w") as f:
        f.write("textid\ttext\ttarget\n")
        i = 0
        for t in pos:
            clean_t = t.replace("\t", " ")
            f.write(f"{i}\t{clean_t}\t1\n")
            i += 1
        for t in neg:
            clean_t = t.replace("\t", " ")
            f.write(f"{i}\t{clean_t}\t0\n")
            i += 1

save_classification_full("test", "test_classification_full")

def save_1_classification_full(split, n_per_class, out_prefix):
    data = ds[split]
    pos = [ex['text'] for ex in data if ex['label'] == 1]
    neg = [ex['text'] for ex in data if ex['label'] == 0]


    os.makedirs("data", exist_ok=True)
    with open(f"data/{out_prefix}.tsv", "w") as f:
        f.write("sentid\tsentence\ttarget\n")
        i = 1
        for t in pos:
            clean_t = t.replace("\t", " ")
            f.write(f"{i}\t{clean_t}\t1\n")
            i += 1
        for t in neg:
            clean_t = t.replace("\t", " ")
            f.write(f"{i}\t{clean_t}\t0\n")
            i += 1
            
save_1_classification_full("test", 1000, "test_classification_1_full")

## Subset vs Entire Dataset

Training and evaluating on the entire dataset yields a noticeable improvement across all performance metrics compared to training on the smaller subset. This is because a larger training dataset provides more diverse examples, helping the model generalize better. It will reduce overfitting to small-sample.